In [7]:
# pip install scikit-learn
# SciKit-Learn é a principal biblioteca de Machine Learning para Python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
# Rodar a Tkinter p/ plotar o painel como no Jupyter
import matplotlib
matplotlib.use('TkAgg')

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


ENDERECO_DADOS = 'https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv'

# obter dados
try:
    print('Obtendo dados de ocorrências...')

    df_ocorrencias = pd.read_csv(ENDERECO_DADOS, sep=';', encoding='iso-8859-1')
    # print(df_ocorrencias.columns) # exibir o nome de todas as colunas

    print(df_ocorrencias.head())    
    print('Dados obtidos com sucesso!')

except Exception as e:
    print('Erro ao obter dados: ', e)
    exit()


Obtendo dados de ocorrências...
   cisp  mes   ano  mes_ano  aisp  risp           munic    mcirc   regiao  \
0     1    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
1     4    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
2     5    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
3     6    1  2003  2003m01     1     1  Rio de Janeiro  3304557  Capital   
4     7    1  2003  2003m01     1     1  Rio de Janeiro  3304557  Capital   

   hom_doloso  ...  cmp  cmba  ameaca  pessoas_desaparecidas  \
0           0  ...  NaN   NaN      21                      2   
1           3  ...  NaN   NaN      15                      6   
2           3  ...  NaN   NaN      47                      2   
3           6  ...  NaN   NaN      26                      2   
4           4  ...  NaN   NaN      10                      1   

   encontro_cadaver  encontro_ossada  pol_militares_mortos_serv  \
0                 0                0                 

In [8]:
# Delimitando as variáveis e a totalizando
try:
    print("inciando a delimitação das variáveis e a totalização...")
    
    df_veiculos = df_ocorrencias[['cisp', 'roubo_veiculo', 'recuperacao_veiculos']]

    # # totalizar o dataframe
    # df_total_veiculos = df_veiculos.groupby('cisp').sum(['roubo_veiculo', 'recuperacao_veiculos']).reset_index()
    # Agrupar e totalizar os roubos e recuperações por CISP no pandas novo   
    df_total_veiculos = df_veiculos.groupby('cisp', as_index=False)[['roubo_veiculo', 'recuperacao_veiculos']].sum()
    
    
    print(df_total_veiculos)

    print('Delimitação e totalização concluídas!')
except Exception as e:
    print("Erro ao delimitar o dataframe: ", e)
    exit()

inciando a delimitação das variáveis e a totalização...
     cisp  roubo_veiculo  recuperacao_veiculos
0       1            572                   541
1       4           2196                  1923
2       5           1362                  1566
3       6           4621                  8086
4       7           2013                  1836
..    ...            ...                   ...
133   159            286                   346
134   165            500                   672
135   166            970                  1510
136   167            106                   409
137   168             62                   167

[138 rows x 3 columns]
Delimitação e totalização concluídas!


In [9]:
# Calculando a Correlação.
try:
    print('Analisando dados...')

    # Excluir uma parte de valores extremos
    df_total_veiculos_cut = (
        df_total_veiculos[
            df_total_veiculos['roubo_veiculo'] < np.percentile(df_total_veiculos['roubo_veiculo'], 95)
        ]
    )
   
    df_total_veiculos_cut = (
        df_total_veiculos_cut[
            df_total_veiculos_cut['recuperacao_veiculos'] < np.percentile(df_total_veiculos_cut['recuperacao_veiculos'], 99)
        ]
    )

    # Transformando em Arrays
    array_roubo_veiculo = np.array(df_total_veiculos_cut['roubo_veiculo'])
    array_recuperacao_veiculos = np.array(df_total_veiculos_cut['recuperacao_veiculos'])

    # Calculando a correlação
    correlacao = np.corrcoef(array_roubo_veiculo, array_recuperacao_veiculos)[0, 1]

    print('Correlação: ', correlacao)

except Exception as e:
    print("Erro ao analisar dados: ", e)
    exit()

Analisando dados...
Correlação:  0.9280535135612554


In [10]:
# Aplicando o modelo de Regressão Linear
# Regressão linear Análise Preditiva
try:
    print("Iniciando a regressão linear...")
    # Instalar a biblioteca scikit-learn
    # SciKit-Learn é a principal biblioteca de Machine Learning para Python
    # pip install scikit-learn

    # Classe para dividir a distribuição dos dados em treino e teste
    from sklearn.model_selection import train_test_split

    # Dividir em treino e teste
    # Roubo de veículos (X): Variável independente - Utilizo para prever
    # Recuperação de veículos (Y): Variável dependente - Prevista
    X_train, X_test, y_train, y_test = train_test_split(
                                            array_roubo_veiculo,
                                            array_recuperacao_veiculos,
                                            test_size=0.2, # tamanho do conjunto de teste, logo o treino terá 0.8
                                            random_state=42
                                        )
    
    # Importar a classe de normalização
    from sklearn.preprocessing import StandardScaler

    scaler = StandardScaler()

    # Normalização (padronização) dos dados de Roubo de Veículos (X)
    # Usa-se o método fit_transform para padronizar os dados de treino.
    # O StandardScaler transforma os dados para que:
    # - a média fique próxima de 0
    # - o desvio padrão fique próximo de 1
    # Isso ajuda o algoritmo a trabalhar com variáveis na mesma escala.

    # O método fit_transform calcula a média e o desvio padrão de X_train
    # e aplica a transformação nos próprios dados de treino, gerenado de uma (1)
    # matriz de 1 coluna e quantas linhas forem necessárias(-1)
    X_train = scaler.fit_transform(X_train.reshape(-1, 1))

    # Dados de teste (X_test)
    # Só é necessário replicar a transformação. Então não usar o método fit_transform
    # usa os mesmos valores de média e desvio padrão que foram calculados
    # no X_train, e aplica a mesma transformação nos dados de X_test.
    X_test = scaler.transform(X_test.reshape(-1, 1))

    # LinearRegression: É responsável por criar o modelo de regressão linear
    from sklearn.linear_model import LinearRegression

    # Criar o modelo linear, ou seja, encontraremos a função y = ax + b
    # Criando a fórmula da regressão linear. 
    # Ainda não sabe quanto vale a e b
    modelo = LinearRegression()

    # Treinar o modelo com os dados de treino y = ax + b
    # O modelo aprende os melhores valores de a e b, com base nos dados de treino
    # No momento do treino, utilizam-se as variáveis de treino de X e y e
    # Descobre a melhor reta, que passa por esses pontos.
    # Ou seja, "aprende uma fórmula matemática para relacionar o roubo de veículos com
    # a recuperação de veículos"
    modelo.fit(X_train, y_train)


    # Avalia a qualidade do modelo com os dados de teste.
    # R² Score (R2 score): Coeficiente de determinação
    # O resultado varia de 0 a 1
    # Quanto mais perto de 1, melhor foi o aprendizado do modelo
    # Acima de 0.7 é um bom modelo.
    # Entre 0.5 e 0.7, é um modelo de qualidade moderada, recomenda-se verificar.
    # Abaixo de 0.5, é uma modelo de qualidade duvidosa
    r2_score = modelo.score(X_test, y_test)
    print('R² Score:', r2_score)

except Exception as e:
    print("Erro ao realizar a regressão linear: ", e)
    exit()

Iniciando a regressão linear...
R² Score: 0.9350070606861218


In [11]:
# Caculando a previsão de recuperação de veículos 40000, 50000, 60000
try:
    # Array com os dados de roubo de veículos para prever a recuperação
    array_roubo_veiculo_pred = np.array([40000, 50000, 60000])

    # Normalizar os dados que serão utilizados para previsão
    array_roubo_veiculo_pred_scaled = scaler.transform(
        array_roubo_veiculo_pred.reshape(-1, 1)
    )

    # Prever a recuperação de veículos - Método modelo.predict()
    recup_pred = modelo.predict(array_roubo_veiculo_pred_scaled)

    print('Previsão de recuperação de veículos (próximos 3 meses): ', recup_pred)

except Exception as e:
    print("Calculando a previsão de recuperação de veículos: ", e)

Previsão de recuperação de veículos (próximos 3 meses):  [23210.19117298 28886.1234366  34562.05570022]


In [ ]:
# Avaliando o Modelo
try:
    print('Avaliando o modelo de previsões...')

    plt.subplots(2, 2, figsize=(15, 5))
    plt.suptitle('Avaliação do modelo de regressão')

    # POSIÇÃO 1: Gráfico de dispersão entre os arrays
    # pip install seaborn
    plt.subplot(2, 2, 1)

    sns.regplot(x=array_roubo_veiculo, y=array_recuperacao_veiculos)
    plt.title('Gráfico de dispersão')
    plt.xlabel('Roubo de veículos')
    plt.ylabel('Recuperação de veículos')

    # Posicionar o texto no início do gráfico
    plt.text(
        min(array_roubo_veiculo),
        max(array_recuperacao_veiculos),
        f'Correlação: {correlacao}',
        fontsize=10
    )


    # POSIÇÃO 2: Gráfico de dispersão entre os dados reais e previsto
    plt.subplot(2, 2, 2)

    # Testar o modelo preditivo nos dados de X_teste
    # Saída na escal original dos dados
    y_pred = modelo.predict(X_test)

    # Retornar os dados de teste para escala real
    X_test = scaler.inverse_transform(X_test)

    # Gráfico de dispersão sem a linha de regressão
    plt.scatter(X_test, y_test, color='blue', label='Dados reais')
    plt.scatter(X_test, y_pred, color = 'red', label='Previsões')

    plt.title('Dados reais x previstos')
    plt.xlabel('Roubo de veículos')
    plt.ylabel('Recuperações de veículos')

    plt.legend()

    # POSIÇÃO 3: Resíduos
    plt.subplot(2, 2, 3)

    # RESÍDUOS: 
    # Diferença entre valores reais e previstos (y_test - y_pred)
    # Análise dos resíduos ajuda a verificar, se o modelo está 
    # capturando bem os padrões dos dados.
    # Bons modelos têm resíduos próximos de 0.
    # Os resíduos devem ser distribuídos aleatoriamente.
    # Se tiverem um padrão, o modelo não é confiável
    residuos = y_test - y_pred

    # plotar em gráfico de dispersão
    plt.scatter(y_pred, residuos)

    # adicioanar uma linha constante no 0
    plt.axhline(y=0, color='black', linewidth=2)

    plt.title('Resíduos')
    plt.xlabel('Previsões')
    plt.ylabel('Resíduos')


    # POSIÇÃO 4: dispersão dos valores simulados
    plt.subplot(2, 2, 4)
    
    plt.scatter(array_roubo_veiculo_pred, recup_pred)

    plt.title('Recuperações de veículos simuladas')
    plt.xlabel('Roubo veículo simulado')
    plt.ylabel('Recuperação de veículo prevista')

    plt.tight_layout()
    plt.show()

except Exception as e:
    print("Erro ao avaliar o modelo: ", e)
    exit()



Avaliando o modelo de previsões...
Erro ao avaliar o modelo:  operands could not be broadcast together with shapes (103,) (26,) 


: 